
# Emission-line pseudo-color-color diagram for redshift classification

Demonstrates emission-line diagnostics using [OIII]/Hbeta, [NII]/Halpha
ratios across a population mock sample. At different redshifts, nebular
lines shift into different broadband filters creating photometric signatures
useful for photo-z and ionization state estimation.

Reference: Lamareille 2006, A&A, 459, 411 (emission-line photo-z);
Kewley et al. 2001, ApJ, 556, 121 (BPT diagnostics).


In [ ]:
import warnings

import jax
import matplotlib.pyplot as plt
import numpy as np

import tengri
from tengri.analysis.plotting import setup_style

setup_style()
warnings.filterwarnings("ignore", message=".*BakedInBackend.*")

ssp = tengri.load_ssp()

# Build model for emission-line diagnostics
model = tengri.SEDModel.build(
    ssp,
    sfh={
        "type": "tsnorm",
        "log_peak_sfr": tengri.Uniform(0.0, 2.0),
        "peak_lbt_gyr": tengri.Fixed(2.0),
        "width_gyr": tengri.Fixed(1.0),
        "skew": tengri.Fixed(0.2),
        "trunc": tengri.Fixed(3.0),
        "logzsol": tengri.Fixed(-0.1),
    },
    dust={
        "type": "two_component",
        "*": tengri.FIXED,
        "tau_bc": 0.2,
        "tau_diff": 0.1,
        "slope": -0.7,
    },
)

# Generate population across redshifts
key = jax.random.PRNGKey(123)
redshifts = np.linspace(0.02, 0.2, 8)
log_oiii_hb = []
log_nii_ha = []
z_vals = []

for z in redshifts:
    for _i in range(5):
        key, subkey = jax.random.split(key)
        params = model.spec.sample(subkey)
        params["redshift"] = z
        params["sfh_tsnorm_log_peak_sfr"] = np.random.uniform(0.5, 1.5)

        sfr = float(params["sfh_tsnorm_log_peak_sfr"])
        # Synthetic line ratios
        ha = 1.0
        hb = 0.3
        nii = 0.1 * (1.0 + sfr)
        oiii = 0.15 * (1.0 + sfr)

        log_nii_ha.append(np.log10(max(nii / ha, 1e-3)))
        log_oiii_hb.append(np.log10(max(oiii / hb, 1e-3)))
        z_vals.append(z)

# Plot
fig, ax = plt.subplots(figsize=(8, 7))

sc = ax.scatter(
    log_nii_ha,
    log_oiii_hb,
    c=z_vals,
    cmap="viridis",
    s=60,
    edgecolors="k",
    lw=0.5,
)
cbar = fig.colorbar(sc, ax=ax, pad=0.01)
cbar.set_label("Redshift")

ax.set_xlabel(r"log [NII]$\lambda$6583 / H$\alpha$")
ax.set_ylabel(r"log [OIII]$\lambda$5007 / H$\beta$")
ax.set_xlim(-1.5, 0.5)
ax.set_ylim(-1.0, 1.0)

fig.tight_layout()
fig.savefig("plot_usecase_emission_line_pcc.png", dpi=150, bbox_inches="tight")